# 05 · Optimization: precision, quantization, KV cache, ONNX export

The toolbox for making a VLM edge-ready, roughly ordered by payoff:

| Technique | Saves | Accuracy risk | On Jetson via |
|---|---|---|---|
| FP32 → FP16/BF16 | 2x memory, much faster on Tensor Cores | ~none | TensorRT `--fp16`, PyTorch `.half()` |
| **Weight-only INT8 / INT4** (W8A16, W4A16) | 2-4x weight memory, ~2-4x faster decode | low / moderate | Edge-LLM, llama.cpp (Q8_0, Q4_K_M), AWQ/GPTQ |
| INT8 weights + activations (W8A8, SmoothQuant) | faster prefill on Tensor Cores | moderate (needs calibration) | TensorRT INT8, Model Optimizer |
| FP8 / NVFP4 | 2-4x | low-moderate | **Thor (Blackwell) only.** Orin (Ampere) has no FP8 Tensor Cores |
| Fewer visual tokens (pixel shuffle, lower res) | prefill compute, KV cache | task-dependent | model design |
| **KV cache** | O(T²) → O(T) decoding | none | every LLM runtime |
| Pruning + distillation (e.g. NVIDIA Minitron) | params | needs retraining | Model Optimizer / NeMo |

Here we apply the first rows to our TinyVLM and measure the effect on **size, speed and robot success**.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import copy, torch
from edge_vlm.train import load_or_train, get_device, VLMPolicy, action_accuracy
from edge_vlm.data import generate_samples, image_to_tensor
from edge_vlm.agent import evaluate_policy
from edge_vlm.optimize import quantize_weights, model_storage_mb, quantization_error, benchmark
from edge_vlm.memory import format_table

device = get_device()
# Loads the notebook-03 checkpoint, or trains one if missing (GPU recommended)
model, tok = load_or_train(ROOT / 'checkpoints' / 'tiny_robot_vlm.pt', device=device)
model = model.float().cpu().eval()
val = generate_samples(150, seed=10_000)
print('device:', device, '| params:', model.param_breakdown()['total'])

## Weight-only quantization, by hand

For each output row (or each group of `g` input weights) we store

$$s = \frac{\max |w|}{2^{b-1}-1}, \qquad q = \text{round}(w / s) \in [-2^{b-1}, 2^{b-1}-1]$$

At runtime $w \approx q \cdot s$. Smaller groups give more scales, which means less error and slightly more storage. INT4 almost always needs groups (32-128). Look at `WeightOnlyQuantLinear` in `src/edge_vlm/optimize.py`, which is about 30 lines.

In [ ]:
variants = {
    'fp32': model,
    'W8 per-channel': quantize_weights(model, bits=8),
    'W4 per-channel': quantize_weights(model, bits=4),
    'W4 group=32': quantize_weights(model, bits=4, group_size=32),
    'W3 group=32': quantize_weights(model, bits=3, group_size=32),
}
rows = []
for name, m in variants.items():
    errs = quantization_error(model, m) if name != 'fp32' else {'-': 0.0}
    acc = action_accuracy(VLMPolicy(m, tok, device=torch.device('cpu')), val)
    rows.append({'variant': name, 'size_MB': round(model_storage_mb(m), 3),
                 'mean_rel_w_err': round(sum(errs.values()) / len(errs), 4),
                 'acc_overall': round(acc['overall'], 3), 'acc_pick': round(acc['pick'], 3)})
print(format_table(rows))

**Read it like an engineer:** INT8 is essentially free. INT4 per-channel may hurt, and grouping usually recovers most of it. Going lower (3-bit) is where things break. For big LMs, smarter algorithms (**AWQ**, **GPTQ**) choose scales using calibration data so they protect the most important weights.

> On a CPU our dequantize-then-matmul is *slower* than FP32. Real speedups come from fused kernels (TensorRT, Edge-LLM, llama.cpp CUDA) that read int4 weights and dequantize in registers. That is why decode gets faster: fewer **bytes** cross the memory bus.

## Does quantization change robot behaviour?

Open-loop accuracy can hide small degradations that compound. Always re-test closed loop after quantizing.

In [ ]:
for name in ['fp32', 'W8 per-channel', 'W4 group=32']:
    stats = evaluate_policy(VLMPolicy(variants[name], tok, device=torch.device('cpu')), n_episodes=100, seed=777)
    print(f'{name:16s}', stats)

## Half precision and latency per stage

We time the three stages of a VLM call separately: **vision encoder** (once per image), **prefill** (all prompt + image tokens) and **decode** (per generated token, using the KV cache). On a GPU, FP16 uses Tensor Cores. On a CPU we use BF16.

In [ ]:
s = val[0]
half = torch.float16 if device.type == 'cuda' else torch.bfloat16
rows = []
for name, dtype in [('fp32', torch.float32), (str(half).split('.')[-1], half)]:
    m = copy.deepcopy(model).to(device=device, dtype=dtype).eval()
    ids = torch.tensor([tok.encode_prompt(s.prompt, m.cfg.num_image_tokens)], device=device)
    px = image_to_tensor(s.image).unsqueeze(0).to(device=device, dtype=dtype)
    with torch.no_grad():
        h = m.merge_embeddings(ids, m.encode_image(px))
        _, past = m.lm_forward(h)
        step = m.embed(ids[:, -1:])
        r = {'precision': name,
             'vision_ms': benchmark(lambda: m.encode_image(px), device=device)['p50_ms'],
             'prefill_ms': benchmark(lambda: m.lm_forward(h), device=device)['p50_ms'],
             'decode_ms/token': benchmark(lambda: m.lm_forward(step, h.shape[1], past), device=device)['p50_ms'],
             'action_ms (cache)': benchmark(lambda: m.generate(ids, px, 6, tok.eos_id, use_cache=True), device=device)['p50_ms'],
             'action_ms (no cache)': benchmark(lambda: m.generate(ids, px, 6, tok.eos_id, use_cache=False), device=device)['p50_ms']}
    rows.append({k: (round(v, 2) if isinstance(v, float) else v) for k, v in r.items()})
print('device:', device)
print(format_table(rows))

For a tiny model, Python and kernel-launch overhead dominate, so the GPU may look barely faster than the CPU. That is exactly why TensorRT (graph fusion, CUDA graphs) and C++ runtimes matter on the Jetson. For big models the memory-bandwidth math from notebook 00 takes over.

## Export to ONNX, the hand-off to TensorRT

We export two graphs, like production VLM runtimes do:
* `vision.onnx`: `pixel_values → image_embeds` (once per camera frame)
* `decoder.onnx`: `input_ids + image_embeds → logits` (per token; no KV cache here, for readability)

then check that ONNX Runtime matches PyTorch numerically.

In [ ]:
from edge_vlm.export import export_onnx, verify_onnx
paths = export_onnx(model, ROOT / 'exports')
for k, p in paths.items():
    print(f'{k:8s} {p}  {p.stat().st_size / 1e6:.2f} MB')
print('max abs error vs PyTorch:', verify_onnx(model, paths))

Copy `exports/` to the Jetson and continue with **notebook 06** (`deploy/jetson/build_trt_engines.sh`).

## Budget check for the real target

Our TinyVLM is ~2.4M params, about 5 MB in FP16, so memory is a non-issue. The point of this exercise is the **workflow**. Plug in the SmolVLM numbers from notebook 04 to see the real constraints:

In [ ]:
from edge_vlm.hardware import get_profile
from edge_vlm.memory import KNOWN_MODELS, ModelSpec, plan_budget

cfg = model.cfg
bd = model.param_breakdown()
tiny = ModelSpec('TinyVLM (ours)', bd['vision_encoder'] + bd['projector'], bd['language_model'] + bd['embeddings (tied)'],
                 cfg.lm_layers, cfg.lm_kv_heads, cfg.lm_dim // cfg.lm_heads, cfg.num_image_tokens)
nano = get_profile('orin_nano_8gb')
rows = [plan_budget(spec, nano, p).as_row() for spec in [tiny, KNOWN_MODELS['SmolVLM-256M'], KNOWN_MODELS['SmolVLM-500M']]
        for p in ['fp16', 'int8', 'int4']]
print(format_table(rows))

### Exercises
1. Quantize *only* the vision encoder vs *only* the LM (`skip=('vision',)` or `skip=('layers',)`). Which part is more sensitive?
2. Implement **asymmetric** quantization (scale + zero-point) in `WeightOnlyQuantLinear`. Does INT4 per-channel improve?
3. Simulate **quantization-aware training**: fine-tune the W4 model for one epoch with a straight-through estimator. Can you recover the lost accuracy?
4. Read the [TensorRT Model Optimizer](https://github.com/NVIDIA/TensorRT-Model-Optimizer) AWQ example and apply it to SmolVLM's LM.